# Near Real-Time Point Observations

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brightbandtech/extremeweatherbench/blob/main/notebooks/near_real_time_point_observations.ipynb)

This notebook demonstrates how to evaluate a forecast against GHCNh station observations. Optimized for Google Colab with a small demo case.

In [ ]:
!pip install -q extremeweatherbench

In [ ]:
import datetime
import extremeweatherbench as ewb
from extremeweatherbench.cases import IndividualCase
from extremeweatherbench.regions import BoundingBoxRegion

# Mini-case: 2020 SW US Heat Wave — Colab-optimized
demo_case = IndividualCase(
    case_id_number=9007,
    title="2020 SW US Heat Wave (demo)",
    start_date=datetime.datetime(2020, 9, 5),
    end_date=datetime.datetime(2020, 9, 8),
    location=BoundingBoxRegion.create_region(
        latitude_min=32.0,
        latitude_max=38.0,
        longitude_min=243.0,
        longitude_max=249.0,
    ),
    event_type="heat_wave",
)
cases = [demo_case]

In [ ]:
forecast = ewb.ZarrForecast(
    source=(
        "gs://weatherbench2/datasets/hres/"
        "2016-2022-0012-1440x721.zarr"
    ),
    name="HRES",
    variable_mapping=ewb.HRES_metadata_variable_mapping,
    storage_options={"remote_options": {"anon": True}},
)

ghcn_target = ewb.GHCN()

In [ ]:
eval_objects = [
    ewb.EvaluationObject(
        event_type="heat_wave",
        metric_list=[
            ewb.metrics.MeanAbsoluteError(
                forecast_variable="surface_air_temperature",
                target_variable="surface_air_temperature",
            ),
            ewb.metrics.MaximumMeanAbsoluteError(
                forecast_variable="surface_air_temperature",
                target_variable="surface_air_temperature",
            ),
        ],
        target=ghcn_target,
        forecast=forecast,
    ),
]

runner = ewb.evaluation(
    case_metadata=cases,
    evaluation_objects=eval_objects,
)
outputs = runner.run_evaluation()

mae = outputs[outputs["metric"] == "MeanAbsoluteError"]
print(mae.groupby("lead_time")["value"].mean())